# Experiment 4 - Testing an LLM with p=Principled Hard-Negative Sampling for RAG Retrieval



![](https://drive.google.com/file/d/16LiDufHtGAf5SbmyJgDONaF2VYf0fvuT/view?usp=sharing)


Files needed to run this:
- config.ini
- noise_impact_summary.tsv
- retrieval_results.json (optional)

In [2]:
!python --version

Python 3.11.13


# IF the python version is higher than 3.11 do run the following cell first:

In [ ]:
# Download and execute set up script
!wget -O py311.sh https://raw.githubusercontent.com/j3soon/colab-python-version/main/scripts/py311.sh
!bash py311.sh

In [ ]:
!sudo update-alternatives --config python3 #when running this select in input python version 3.10/3.11 out of the options

In [3]:
!python --version

Python 3.11.13


### Setup from source

**Step 1.** Clone the repo


**Step 2.** Create a conda environment
```bash
conda create -n bcqa



**Step 3.**
```bash
pip install -e dexter-cqa



## Setup using pip
Alternately you can also use pip

```bash
pip install dexter-cqa


In [4]:
pip install dexter-cqa

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 68.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of sentence-transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of sentence-transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 13.7 MB/s et

In [ ]:
#if needed to restart colab run this instead of restart button:
import os
os.kill(os.getpid(), 9)

In [ ]:
import json
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer
from random import sample, shuffle
from tqdm import tqdm

from dexter.config.constants import Split
from dexter.data.loaders.RetrieverDataset import RetrieverDataset

print("All packages imported successfully")

In [ ]:
#!nvidia-smi

Fri Jan 16 18:13:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
!python --version

Python 3.11.13





# Dataset Description

Dexter extends the several well-known datasets into an open-domain setting. *Dexter* contains 7 datasets with over 210,000 QA pairs, from corpus sizes varying from 0.5M to 25M documents. Details of the datasets are given below.

|  Dataset Name  |  Dataset alias |                  Homepage                 |                Characteristics               | #Questions | Corpus Size |
|:--------------:|:--------------:|:-----------------------------------------:|:--------------------------------------------:|:----------:|:-----------:|
| MusiqueQA      | musiqueqa      | [Link](https://github.com/StonyBrookNLP/musique)  | Connected multi-hop reasoning                |    16.8        | 570k        |
| WikiMultiHopQA | wikimultihopqa | [Link](https://github.com/Alab-NII/2wikimultihop) | Comparative multi-hop reasoning              | 190k       | 570k        |
| StrategyQA     | strategyqa     | [Link](https://allenai.org/data/strategyqa)       | Multi-hop reasoning, Implicit Reasoning      | 2.7k       | 26.6M       |
| AmbigQA        | ambignq        | [Link](https://nlp.cs.washington.edu/ambigqa/)    | Ambiguous Questions                          | 12k        | 24.3M       |
| OTT-QA         | ottqa          | [Link](https://ott-qa.github.io/)                 | Table and Text multi-hop reasoning           | 2.1k       | 6.5M        |
| TAT-QA         | tatqa          | [Link](https://nextplusplus.github.io/TAT-QA/)    | Financial Table and Text multi-hop reasoning | 2.9k       | 7000        |
| FinQA          | finqa          | [Link](https://github.com/czyssrs/FinQA)          | Financial Table and Text multi-hop reasoning | 8k         | 24.8k       |



for the project to work, you need a hugging face account (follow the tutorial):

Authenticate with Hugging Face
Step 1: Get a Hugging Face Token

Go to https://huggingface.co/settings/tokens
Create a new token (or use an existing one)
Copy the token

Step 2: Accept the Llama License

Go to https://huggingface.co/meta-llama/Llama-2-7b-chat-hf
Click "Agree and access repository"
Fill out the form and accept Meta's license terms

In [ ]:
print("Authenticating with Hugging Face...")
from huggingface_hub import notebook_login
notebook_login()
print("✓ Authentication complete\n")

In [ ]:
class LlamaEngine:
    """
    Wrapper for Llama-2 model to generate answers.
    """

    def __init__(self, model_name="meta-llama/Llama-2-7b-chat-hf",
                 temperature=0.3, max_new_tokens=256):
        print(f"Loading {model_name}...")
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.temperature = temperature
        self.max_new_tokens = max_new_tokens

    
        self.pipeline = transformers.pipeline(
            "text-generation",
            model=self.model_name,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        print("✓ Model loaded\n")

    def get_llama_completion(self, system_prompt: str, user_prompt: str):
        """
        Generate an answer using Llama-2 chat format.
        """
        
        prompt = (
            "<s>[INST] <<SYS>>\n"
            f"{system_prompt.strip()}\n"
            "<</SYS>>\n\n"
            f"{user_prompt.strip()} [/INST]"
        )

        outputs = self.pipeline(
            prompt,
            max_new_tokens=self.max_new_tokens,
            do_sample=True,
            num_return_sequences=1,
            temperature=self.temperature,
            top_k=10,
            top_p=0.95
        )

        return outputs[0]["generated_text"]


def evaluate_answer(predicted, gold_answer):
    """
    Check if the predicted answer matches the gold answer.
    """
    pred_lower = predicted.lower()

    
    if "not possible" in pred_lower or "unknown" in pred_lower:
        return 0

    
    if "[Final Answer]:" in predicted:
        pred_answer = predicted.split("[Final Answer]:")[-1].strip()
    else:
        
        pred_answer = predicted.strip()

    
    if gold_answer.lower() in pred_answer.lower():
        return 1  # CORRECT
    else:
        return 0  # WRONG


print("="*70)
print("LOADING DATA")
print("="*70)

print("\n[1/3] Loading retrieval results...")
try:
    with open("/content/retrieval_results.json") as f:
        evidence = json.load(f)
    print(f"Loaded retrieval results for {len(evidence)} questions")
except FileNotFoundError:
    print("ERROR: retrieval_results.json not found!")
    print("\nYou need to either:")
    print("  1. Upload the file to /content/")
    print("  2. Or run retrieval yourself")
    raise

print("\n[2/3] Loading dataset...")
loader = RetrieverDataset(
    "wikimultihopqa",
    "wiki-musiqueqa-corpus",
    "config.ini",
    Split.DEV
)

queries, qrels, corpus = loader.qrels()
raw_data = loader.base_dataset.raw_data

print(f"Loaded {len(raw_data)} questions")
print(f"Loaded {len(corpus)} documents in corpus")


print("\n[3/3] Initializing LLM...")
llm_instance = LlamaEngine(
    model_name="meta-llama/Llama-2-7b-chat-hf",
    temperature=0.3,
    max_new_tokens=256
)

In [ ]:
# Check what's loaded
try:
    print(f"LLM loaded: {llm_instance.model_name}")
except:
    print("Need to load LLM")

try:
    print(f"Corpus loaded: {len(corpus)} documents")
except:
    print("Need to load corpus")

try:
    print(f"Raw data loaded: {len(raw_data)} questions")
except:
    print("Need to load dataset")

try:
    print(f"Retrieval results loaded: {len(evidence)} questions")
except:
    print("Need to load retrieval_results.json")

In [ ]:
import pickle
import json
from random import shuffle, sample

print("[INFO] Imports: standard libs loaded")

from dexter.config.constants import Split
print("[INFO] dexter.config.constants loaded")

from dexter.data.loaders.RetrieverDataset import RetrieverDataset
print("[INFO] RetrieverDataset loaded")

from dexter.retriever.dense.Contriever import Contriever
print("[INFO] Contriever model class loaded")

from dexter.utils.metrics.SimilarityMatch import DotScore
print("[INFO] DotScore metric loaded")

from dexter.data.datastructures.hyperparameters.dpr import DenseHyperParams
print("[INFO] DenseHyperParams loaded")

print("\n[SUCCESS] All imports completed successfully.")

Outcomment the following cell if you want to run the hard_negatives retriever yourself

In [ ]:
# # os.environ["HF_HUB_OFFLINE"] = "1"
# loader = RetrieverDataset("wikimultihopqa",
#                           "wiki-musiqueqa-corpus",
#                           "config.ini",
#                           Split.TRAIN,
#                           tokenizer=None)
# queries, qrels, corpus = loader.qrels()
# config_instance = DenseHyperParams(
#     query_encoder_path="facebook/contriever",
#     document_encoder_path="facebook/contriever",
#     batch_size=64,
#     show_progress_bar=True
# )
# retriever = Contriever(config_instance)
# similarity_measure = DotScore()

# corpus_map = {doc.id(): doc for doc in corpus} # info for each passage
# queries_map = {query.id(): query for query in queries} # id and text of each question
# all_augmented_contexts = [] # positive passages + hard negatives
# dpr_entries = [] # DPR entries used for ADORE-tuning later
# num_hard_negs = 3
# num_negs = 2
# broad_results = retriever.retrieve(corpus, queries, 100, similarity_measure, chunk=True, chunksize=50000)

# for query_id, retrieved_docs in broad_results.items():
#     gold_doc_ids = set(qrels[query_id].keys())
#     ranked_docs = list(retrieved_docs.items())
#     positives = []
#     hard_negs = []
#     negs = []

#     for doc_id, score in ranked_docs:
#         doc_obj = corpus_map.get(doc_id)
#         if not doc_obj:
#             continue

#         doc_entry = {
#             "id": doc_id,
#             "title": doc_obj.title(),
#             "text": doc_obj.text(),
#             "score": score
#         }

#         if doc_id in gold_doc_ids:
#             positives.append(doc_entry)
#         else:
#             negs.append(doc_entry)

#     negs.sort(key=lambda x: x["score"], reverse=True)
#     hard_negs = negs[:num_hard_negs]
#     remaining_negs = negs[num_hard_negs:]
#     if len(remaining_negs) >= num_negs:
#         random_negs = sample(remaining_negs, num_negs)
#     else:
#         random_negs = remaining_negs

#     selected_docs = positives + hard_negs
#     shuffle(selected_docs)
#     all_augmented_contexts.append(selected_docs)

#     dpr_entry = {
#         "id": query_id,
#         "question": queries_map[query_id].text(),
#         "positive_ctxs": [
#             {"title": p["title"], "text": p["text"], "passage_id": p["id"]}
#             for p in positives
#         ],
#         "hard_negative_ctxs": [
#             {"title": hn["title"], "text": hn["text"], "passage_id": hn["id"]}
#             for hn in hard_negs
#         ],
#         "negative_ctxs": [
#             {"title": rn["title"], "text": rn["text"], "passage_id": rn["id"]}
#             for rn in random_negs
#         ]
#     }
#     dpr_entries.append(dpr_entry)

# with open("train_dpr_ready.jsonl", "w", encoding="utf-8") as f:
#     for entry in dpr_entries:
#         f.write(json.dumps(entry) + "\n")

# # read
# # with open("all_augmented_contexts.pkl", "rb") as f:
# #     all_augmented_contexts = pickle.load(f)

# # save
# with open('all_augmented_contexts.pkl', 'wb') as f:
#   pickle.dump(all_augmented_contexts, f)

In [ ]:
from random import shuffle
from tqdm import tqdm
import pandas as pd

k_relevant = 3

system_prompt = (
    "Follow the given examples and Given the question and context "
    "output final answer for the question using information in the "
    "context and give answer in form of [Final Answer]: \n"
)

few_shot = """[Question]: When does monsoon season end in the state the area code 575 is located?
[Final Answer]: mid-September.
[Question]: The birth country of Jayantha Ketagoda left the British Empire when?
[Final Answer]: February 4, 1948.\n\n"""

performance_summary = []

for num_hard_negs in [1, 2, 3]:  # Test with 1,2,3 hard negatives
    
    print(f"\n{'='*70}")
    print(f"Testing: {k_relevant} relevant + {num_hard_negs} hard negatives")
    print(f"{'='*70}")

    matches = 0
    total = 0
    log = {"questions": [], "answers": [], "gold": [], "correct": []}

    for row in tqdm(raw_data[:1200], desc=f"Hard negs={num_hard_negs}"):
        q_id = row.question.id()

        if q_id not in evidence:
            continue

        gold_ids = set(qrels[q_id].keys())
        retrieved = list(evidence[q_id].items())

        positives = [(d, s) for d, s in retrieved if d in gold_ids]
        negatives = [(d, s) for d, s in retrieved if d not in gold_ids]

        top_pos = positives[:k_relevant]
        hard_negs = negatives[:num_hard_negs]  

        pos_texts = [corpus[int(d)].text() for d, _ in top_pos]
        neg_texts = [corpus[int(d)].text() for d, _ in hard_negs]

        all_texts = pos_texts + neg_texts
        shuffle(all_texts)
        context = " ".join(all_texts)

        prompt = few_shot + f"Evidence: {context} \nQuestion: {row.question.text()}"
        answer = llm_instance.get_llama_completion(system_prompt, prompt)

        gold = row.answer.text().strip().lower()
        pred = (answer.split("[Final Answer]:")[-1] if "[Final Answer]:" in answer else answer).strip().lower()

        if "not possible" not in answer.lower() and "unknown" not in answer.lower() and gold in pred:
            matches += 1

        total += 1
        log["questions"].append(row.question.text())
        log["answers"].append(answer)
        log["gold"].append(row.answer.text())
        log["correct"].append(1 if gold in pred else 0)

    em = matches / total

    print(f"\nResults for {num_hard_negs} hard negatives:")
    print(f"  EM: {em:.4f} ({matches}/{total})")

    performance_summary.append({
        "hard_neg_count": num_hard_negs,
        "em_score": em,
        "correct": matches,
        "total": total
    })

    pd.DataFrame(log).to_csv(f"hard_negatives_{num_hard_negs}_results.tsv", sep="\t", index=False)
    print(f"  ✓ Saved: hard_negatives_{num_hard_negs}_results.tsv")

# Save summary
summary_df = pd.DataFrame(performance_summary)
summary_df.to_csv("hard_negatives_summary.tsv", sep="\t", index=False)

print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(summary_df.to_string(index=False))
